In [1]:
!pip install torch torchvision torchaudio bitsandbytes transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 8.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset
import torch
#from bitsandbytes import quantization_config

# Загружаем модель и токенизатор
model_name = "EleutherAI/gpt-j-6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Применяем квантизацию
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,  # Включаем квантизацию 4-бит
    device_map="auto"
)

from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/930 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


pytorch_model.bin:   0%|          | 0.00/24.2G [00:00<?, ?B/s]

Some weights of the model checkpoint at EleutherAI/gpt-j-6B were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bias', 'transformer.h.17.attn.masked_bias', 'transformer.h.18.attn.bias', 'transformer.h.18.attn.masked_bias', 'transformer.h.19.attn.bias', 'transformer.h.19.attn.masked_bias', 'transformer.h.2.attn.bias', 'transformer.h.2.attn.masked_bias', 'transformer.h.20.attn.bi

In [13]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# Конфигурация LoRA адаптеров
lora_config = LoraConfig(
    r=8,  # Размер адаптера
    lora_alpha=16,  # Коэффициент для LoRA
    lora_dropout=0.1  # Вероятность выпадения
)

# Получаем модель с LoRA адаптерами
peft_model = get_peft_model(model, lora_config)

In [24]:

# Функция токенизации
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="longest", max_length=512)

# Загружаем датасет
dataset = load_dataset("text", data_files={"train": "all_songs.txt"})

# Проверим, если токен паддинга уже существует
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Используем токен конца последовательности для паддинга

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Data collator (нужен для Trainer)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, без MLM
)

# Параметры обучения с сохранением по шагам и увеличенной частотой обновлений
training_args = TrainingArguments(
    output_dir="./gpt-j-slava-kpss",
    per_device_train_batch_size=1,  # Один элемент в батче
    gradient_accumulation_steps=8,  # Суммируем градиенты несколько шагов для оптимизации использования памяти
    num_train_epochs=1,  # Один цикл (попробуйте несколько, если потребуется)
    save_strategy="steps",  # Сохраняем модель по шагам
    save_steps=500,  # Сохраняем каждые 500 шагов (можете настроить это значение)
    save_total_limit=1,  # Храним только последнюю модель
    logging_strategy="no",  # Логируем по шагам
    report_to="none",  # Не отправляем метрики
    fp16=True,  # Используем 16-битную точность
    weight_decay=0.01,
)



# Запускаем обучение
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator
)

trainer.train()


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

KeyError: "Column train not in the dataset. Current columns in the dataset: ['text', 'input_ids', 'attention_mask']"

In [22]:
# Устанавливаем модель в режим оценки (inference)
model.eval()

# Генерация текста, настроенная на создание песен
def generate_song_line(prompt, max_length=100):
    # Токенизируем введенный текст
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True)

    # Перемещаем входные данные на тот же девайс, что и модель
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    # Генерация текста с использованием модели с параметрами, подходящими для песен
    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],  # Входные токены
            max_length=max_length,  # Максимальная длина выходной последовательности
            num_return_sequences=1,  # Количество сгенерированных вариантов текста
            no_repeat_ngram_size=2,  # Убираем повторяющиеся фразы
            top_k=50,  # Ограничение на количество наиболее вероятных токенов для выбора
            top_p=0.95,  # Используем сэмплирование по вероятности
            temperature=0.8,  # Контролируем креативность модели
            pad_token_id=tokenizer.pad_token_id,  # Идентификатор токена паддинга
            eos_token_id=tokenizer.eos_token_id,  # Идентификатор токена конца строки
            do_sample=True,  # Включаем сэмплирование для большей разнообразности
        )

    # Декодируем сгенерированные токены обратно в текст
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Можно добавить разбиение на строки для имитации куплетов/припевов
    song_lines = generated_text.split(". ")  # Разделяем по точкам, чтобы строки напоминали куплеты
    return song_lines

# Пример использования
prompt = "Я иду по улице"
generated_song = generate_song_line(prompt)
for line in generated_song:
    print(line)



Я иду по улице Кашавака
На месте сидит домик винограда чи камбала, запрокутил, тонкие,
клейм
